In [3]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [7]:
"""
Pipeline encodage
"""

import os
import pandas as pd
import category_encoders as ce
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# =============================================================================
# 0. Définition des chemins (compatibles GitHub)
# =============================================================================

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Lecture depuis :", DATA_DIR)
print("Fichiers trouvés :", os.listdir(DATA_DIR))

# Chargement du fichier ZIP contenant le CSV
df = pd.read_csv(os.path.join(DATA_DIR, "accidents_France_raw.zip"))

# =============================================================================
# 1. Feature engineering temporel : tranche horaire
# =============================================================================

def categoriser_heure(h: int) -> str:
    if 6 <= h < 12:
        return "Matin"
    elif 12 <= h < 18:
        return "Apres-midi"
    elif 18 <= h < 22:
        return "Soir"
    return "Nuit"

heure = df["hrmn"].astype(str).str.replace(":", "").str.zfill(4).str[:2].astype(int)
df["TRANCHE_HORAIRE"] = heure.apply(categoriser_heure)

ORDRE_HORAIRE = {"Matin": 1, "Apres-midi": 2, "Soir": 3, "Nuit": 4}
df["tranche_horaire_ord"] = df["TRANCHE_HORAIRE"].map(ORDRE_HORAIRE)

# =============================================================================
# 2. Suppression des colonnes redondantes ou inutiles
# =============================================================================

COLS_A_SUPPRIMER = [
    "jour", "mois", "date_time", "hrmn", "an",
    "equipement_secu_3", "action_pieton",
    "lat", "long",
]
df.drop(columns=COLS_A_SUPPRIMER, inplace=True, errors="ignore")

# =============================================================================
# 3. Nettoyage minimal AVANT split
# =============================================================================

df = df.dropna(subset=["type_collision", "luminosite", "meteo"])
df["equipement_secu_2"] = df["equipement_secu_2"].fillna(0)

# =============================================================================
# 4. Split AVANT toute imputation dépendante des distributions
# =============================================================================

X = df.drop(columns=["gravite"])
y = df["gravite"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

df_train = X_train.copy()
df_train["gravite"] = y_train
df_test = X_test.copy()
df_test["gravite"] = y_test

# =============================================================================
# 5. Imputation du mode sur le train uniquement
# =============================================================================

COLS_A_IMPUTER_MODE = ["regime_circulation", "motif_deplacement", "sexe_usager"]

modes_train = {col: df_train[col].mode()[0] for col in COLS_A_IMPUTER_MODE}

for col in COLS_A_IMPUTER_MODE:
    df_train[col] = df_train[col].fillna(modes_train[col])
    df_test[col] = df_test[col].fillna(modes_train[col])

# Nettoyage final des colonnes critiques
COLS_A_NETTOYER = ["equipement_secu_1", "nb_voies", "amenagement"]
df_train = df_train.dropna(subset=COLS_A_NETTOYER)
df_test = df_test.dropna(subset=COLS_A_NETTOYER)

print("Train NA restants :", df_train.isna().sum().sum())
print("Test NA restants  :", df_test.isna().sum().sum())
print("Train lignes :", len(df_train))
print("Test lignes  :", len(df_test))

# =============================================================================
# 6. Regroupement des types de véhicules
# =============================================================================

MAPPING_VEHICULE = {
    7.0: "Voiture",
    10.0: "Poids_Lourd_Utilitaire", 13.0: "Poids_Lourd_Utilitaire",
    14.0: "Poids_Lourd_Utilitaire", 17.0: "Poids_Lourd_Utilitaire",
    1.0: "Velo_Trotinette", 80.0: "Velo_Trotinette", 50.0: "Velo_Trotinette",
    2.0: "Deux_Roues_Moteur", 30.0: "Deux_Roues_Moteur",
    33.0: "Deux_Roues_Moteur", 32.0: "Deux_Roues_Moteur",
    37.0: "Transport_Commun", 38.0: "Transport_Commun",
}
df_train["type_vehicule_simplifie"] = df_train["type_vehicule"].map(MAPPING_VEHICULE).fillna("Autre")
df_test["type_vehicule_simplifie"] = df_test["type_vehicule"].map(MAPPING_VEHICULE).fillna("Autre")

# =============================================================================
# 7. Label Encoding
# =============================================================================

df_train["sexe_encoded"] = df_train["sexe_usager"].map({1.0: 0, 2.0: 1})
df_test["sexe_encoded"] = df_test["sexe_usager"].map({1.0: 0, 2.0: 1})

for col in ["localisation_agglo", "is_weekend", "is_ferie"]:
    if col in df_train.columns:
        df_train[col] = df_train[col].astype(int)
        df_test[col] = df_test[col].astype(int)

# =============================================================================
# 8. Target Encoding (train uniquement)
# =============================================================================

TARGET_ENCODING_CONFIG = {
    "equipement_secu_2": "secu2_encoded",
    "equipement_secu_1": "secu_encoded",
    "motif_deplacement": "motif_encoded",
    "obstacle_mobile": "obsm_encoded",
    "point_choc": "choc_encoded",
    "trace_plan": "trace_encoded",
    "obstacle_fixe": "obs_fixe_encoded",
    "manœuvre_avant_accident": "manoeuvre_encoded",
    "type_motorisation": "moteur_encoded",
    "regime_circulation": "regime_encoded",
    "nb_voies": "voies_encoded",
    "profil_route": "profil_encoded",
    "type_collision": "collision_encoded",
    "categorie_route": "route_encoded",
    "intersection": "intersection_encoded",
    "meteo": "meteo_encoded",
    "etat_surface": "surface_encoded",
    "amenagement": "amenagement_encoded",
    "situation_accident": "situation_encoded",
    "voie_reservee": "voie_reservee_encoded",
    "sens_circulation": "sens_circulation_encoded",
}

for col_source, col_encodee in TARGET_ENCODING_CONFIG.items():
    encoder = ce.TargetEncoder(cols=[col_source], smoothing=10)
    encoder.fit(df_train[col_source], df_train["gravite"])
    df_train[col_encodee] = encoder.transform(df_train[col_source])
    df_test[col_encodee] = encoder.transform(df_test[col_source])

# =============================================================================
# 9. One-Hot Encoding
# =============================================================================

COLS_ONE_HOT = ["categorie_usager", "type_vehicule_simplifie"]

df_train = pd.get_dummies(df_train, columns=COLS_ONE_HOT, drop_first=False)
df_test = pd.get_dummies(df_test, columns=COLS_ONE_HOT, drop_first=False)
df_test = df_test.reindex(columns=df_train.columns, fill_value=0)

# =============================================================================
# 10. Nettoyage final
# =============================================================================

COLS_A_RETIRER = [
    "TRANCHE_HORAIRE",
    *TARGET_ENCODING_CONFIG.keys(),
    "sexe_usager",
    "type_vehicule",
]

df_train = df_train.drop(columns=COLS_A_RETIRER, errors="ignore")
df_test = df_test.drop(columns=COLS_A_RETIRER, errors="ignore")

X_train = df_train.drop(columns=["gravite"])
y_train = df_train["gravite"]
X_test = df_test.drop(columns=["gravite"])
y_test = df_test["gravite"]

print(X_train.info())
print(f"Train : {X_train.shape[0]} lignes — Test : {X_test.shape[0]} lignes")

# =============================================================================
# 11. Sauvegarde des jeux encodés
# =============================================================================

X_train.to_csv(os.path.join(OUTPUT_DIR, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(OUTPUT_DIR, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(OUTPUT_DIR, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(OUTPUT_DIR, "y_test.csv"), index=False)

# =============================================================================
# 12. Pipeline de modélisation
# =============================================================================

NUM_COLS = ["vitesse_max", "age_usager"]
CAT_COLS = [c for c in X_train.columns if c not in NUM_COLS]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_COLS),
        ("cat", "passthrough", CAT_COLS),
    ]
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=1000)),
])

# model.fit(X_train, y_train)
# model.score(X_test, y_test)


Lecture depuis : C:\Users\hp\Documents\Projet_accidents\data
Fichiers trouvés : ['accidents_France_encoded.zip', 'accidents_France_raw.zip', 'X_test.zip', 'X_train.zip', 'y_test.zip', 'y_train.zip']
Train NA restants : 4412
Test NA restants  : 1126
Train lignes : 574672
Test lignes  : 143613
<class 'pandas.core.frame.DataFrame'>
Index: 574672 entries, 426016 to 121960
Data columns (total 38 columns):
 #   Column                                          Non-Null Count   Dtype  
---  ------                                          --------------   -----  
 0   luminosite                                      574672 non-null  float64
 1   localisation_agglo                              574672 non-null  int64  
 2   vitesse_max                                     574672 non-null  int64  
 3   age_usager                                      574672 non-null  int64  
 4   is_weekend                                      574672 non-null  int64  
 5   is_ferie                                     